# Install Some Libraries

In [1]:
!pip -qq install datasets sentencepiece transformers[torch] evaluate sacrebleu rouge_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 7.2 MB/s eta 0:00:00


# Data Preparation

## Download Dataset: PhoMT_Partial
This data contains aligned text pairs in Vietnamese and English. The dataset is extracted from PhoMT. For full dataset, visit: [vietgpt/phomt](https://huggingface.co/datasets/vietgpt/phomt).


We will download a partially pre-processed, publicly available version of this dataset from [Huggingface](https://huggingface.co/datasets/Darejkal/phomt_partial).




In [2]:
from datasets import load_dataset,Dataset

dataset = load_dataset(
    path="Darejkal/phomt_partial",
    num_proc=2,
    split="train",
    verification_mode="no_checks",
    cache_dir="cache")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Setting num_proc from 2 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Setting num_proc from 2 back to 1 for the test split to disable multiprocessing as it only contains one shard.


Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [3]:
dataset

Dataset({
    features: ['en', 'vi'],
    num_rows: 2000
})

In [4]:
data_df = dataset.to_pandas()
data_df

,en,vi
0,It begins with a countdown.\n,Câu chuyện bắt đầu với buổi lễ đếm ngược.\n
1,"On August 14th, 1947, a woman in Bombay goes i...","Ngày 14, tháng 8, năm 1947, gần nửa đêm, ở Bom..."
2,"Across India, people hold their breath for the...","Cùng lúc, trên khắp đất Ấn, người ta nín thở c..."
3,"And at the stroke of midnight, a squirming inf...","Khi đồng hồ điểm thời khắc nửa đêm, một đứa tr..."
4,"These events form the foundation of ""Midnight'...","Những sự kiện này là nền móng tạo nên ""Những đ..."
...,...,...
1995,"Now, this is not only about numbers; this is n...","Bây giờ, vấn đề không chỉ ở con số không chỉ ở..."
1996,The majority of the environmental impacts on t...,Phần lớn các tác động môi trường trên hành tin...
1997,"The majority of the planet, aspiring for devel...","Phần lớn của hành tinh, khao khát phát triển, ..."
1998,"The second pressure on the planet is, of cours...",Áp lực thứ hai tất nhiên là vấn đề khí hậu - -...


In [5]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   en      2000 non-null   object
 1   vi      2000 non-null   object
dtypes: object(2)
memory usage: 31.4+ KB


# Load model for inference

We wil load model and tokenizer from huggingface and test inferencing on our data

In [21]:
from transformers import AutoTokenizer, MT5ForConditionalGeneration, AutoModelForSeq2SeqLM

import torch
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small", cache_dir="cache",)
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small", cache_dir="cache", legacy=False, use_fast=False)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [8]:
data = load_dataset(
    path="Darejkal/phomt_partial",
    num_proc=2,
    verification_mode="no_checks",
    cache_dir="cache")

input_data = data['train']
test_data = data['test']

In [9]:
print(data)

DatasetDict({
    train: Dataset({
        features: ['en', 'vi'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['en', 'vi'],
        num_rows: 1000
    })
})


In [10]:
query = input_data[0]['en']
ids = torch.tensor([tokenizer.encode(query)])
print(query)
print(ids)

It begins with a countdown.

tensor([[  1385,  10672,    263,    514,    259,    262,    259, 235045,    260,
              1]])


### Test model generation

Note: mT5 was only pre-trained on mC4 excluding any supervised training. Therefore, this model has to be fine-tuned before it is useable on a downstream task.

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# query = input_data[10]['en']
query = "Hanoi University of <extra_id_0> (HUST), or Trường Đại học Bách khoa Hà Nội, is one of the leading technical universities in Vietnam. Established in 1956, it was the first and largest university in engineering and technology in the country. HUST plays a significant role in producing high-quality human resources for Vietnam's industrial and technological development."
input_ids = tokenizer(query,return_tensors="pt")

output_vec = model.generate(**input_ids)
output=tokenizer.batch_decode(output_vec, skip_special_tokens=True)
print(output)

['<extra_id_0> Technology <extra_id_1> (HUST) HUST <extra_id_2> (HUST)']


### Free up GPU

In [12]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

# Finetuning for translation task

We will train mT5 for translation task from English to Vietnamese. We will use the prefix prompt to formulate the translation task as a text2text problem. For a list of prompt, please refer to `Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer`.

In [13]:
MAX_TOKEN_LENGTH = 1024
PHOMT_PREFIX = "translate English to Vietnamese: {text}"
DATA_SOURCE = "Darejkal/phomt_partial"
CACHE_DIR= " cache"
OUT_FOLDER = "out"
IGNORE_INDEX = -100
RESPONSE_OUTFILE = "test_answers.txt"

In [14]:
from datasets import load_dataset
data = load_dataset(
    path=DATA_SOURCE,
    num_proc=2,
    verification_mode="no_checks",
    cache_dir=CACHE_DIR)
input_data = data['train']
input_data = input_data.select(range(50))
test_data = data['test']

Setting num_proc from 2 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Setting num_proc from 2 back to 1 for the test split to disable multiprocessing as it only contains one shard.


Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [15]:
from transformers import AutoTokenizer
from functools import partial
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small", cache_dir="cache", legacy=False, use_fast=False)
custom_tokenize = partial(tokenizer, max_length=MAX_TOKEN_LENGTH, padding="max_length", truncation=True)
def preprocess_function(examples):
    inputs, answers=list(zip(*[(PHOMT_PREFIX.format(text=examples["en"][i]), examples["vi"][i]) for i in range(len(examples["en"]))]))
    model_inputs = custom_tokenize(text=inputs, text_target=answers)
    return model_inputs

tokenized_datasets = input_data.map(preprocess_function, batched=True, num_proc=2)
tds = tokenized_datasets.train_test_split(10)
tokenized_train_datasets = tds['train']
tokenized_eval_datasets = tds['test']

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

### Target score
We will use sacrebleu as the target score for the translation task.

Scarebleu: Higher is better


In [16]:
import evaluate
import numpy as np
from transformers.trainer_utils import EvalPrediction
eval_metrics = evaluate.combine(["sacrebleu"])
def compute_metrics(eval_preds):
    preds, labels = eval_preds

    preds = np.where(preds != IGNORE_INDEX, preds, tokenizer.pad_token_id)
    labels = np.where(labels != IGNORE_INDEX, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = eval_metrics.compute(predictions=decoded_preds,
                                  references=decoded_labels)
    result = {"bleu": result["score"]}
    return result

In [17]:
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small", cache_dir=CACHE_DIR, device_map="auto")

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [18]:
from transformers import Seq2SeqTrainingArguments
training_args = Seq2SeqTrainingArguments(
    report_to="none",
    output_dir="/kaggle/working",
    save_steps=100,
    eval_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    evaluation_strategy="steps",
    seed=117,
    learning_rate=2e-4,
    per_device_eval_batch_size=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    lr_scheduler_type="cosine",
    weight_decay=1e-4,
    max_grad_norm=1.0,
    warmup_ratio=0.01,
    group_by_length=False,
    num_train_epochs=1,
    predict_with_generate=True,
    dataloader_num_workers=4,
    dataloader_prefetch_factor=2,
    generation_num_beams=5,
    generation_max_length=MAX_TOKEN_LENGTH,
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [19]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer,
                                       model=model,
                                       pad_to_multiple_of=8,
                                       label_pad_token_id=IGNORE_INDEX)

In [20]:
from transformers import Seq2SeqTrainer
trainer =  Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_datasets,
    eval_dataset=tokenized_eval_datasets,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


KeyboardInterrupt: 

### Evaluate

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def generate_answer(texts):
    input_ids = tokenizer(texts, return_tensors="pt", max_length=1024, padding=True, truncation=True, return_token_type_ids=False).to(device)
    output_ids= model.generate(**input_ids,
                            num_return_sequences=1,
                            num_beams=5,
                            early_stopping=True,
                            max_length=MAX_TOKEN_LENGTH,
                            )
    return tokenizer.batch_decode(output_ids, skip_special_tokens=True)

In [ ]:
import datasets
ds = test_data.select(range(10))
index = 0

writer = open(RESPONSE_OUTFILE, "w",encoding="utf-8")

while index < len(ds):
    next=min(index + 4,len(ds))
    texts=[PHOMT_PREFIX.format(text=ds[i]["en"])for i in range(index,next)]
    outputs = generate_answer(texts)
    for output in outputs:
        print(output.strip().replace("\n"," "))
        writer.write(output.strip().replace("\n"," ") + "\n")
    index = next

writer.close()

In [ ]:
import evaluate
import json
references = test_data['vi'][
predictions = [
    line.strip() for line in open(RESPONSE_OUTFILE, "r",encoding="utf-8").readlines()
]

sacrebleu = evaluate.load("sacrebleu")
results = sacrebleu.compute(predictions=predictions, references=references)
print(results)
with open(RESPONSE_OUTFILE+"_blue","w") as f:
    f.write(json.dumps(results))


rougue = evaluate.load("rouge")
results = rougue.compute(predictions=predictions, references=references)
print(results)
with open(RESPONSE_OUTFILE+"_rougue","w") as f:
    f.write(json.dumps(results))